In [ ]:
import decoupler as dc
from scipy.stats import rankdata
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix, issparse
from scipy import sparse
from tqdm.auto import tqdm
from scipy.special import ndtr
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Generate simulated data
!uv run simulated_data_generator.py simulated_data_parameters.json

In [ ]:
# Gene Expression data
raw = pd.read_csv("simulated_data/simulated_scRNASeq_data.tsv", sep="\t", index_col=0)
adata = sc.AnnData(csr_matrix(raw.values), var=pd.DataFrame(index=raw.columns), obs=pd.DataFrame(index=raw.index))
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# Prior Knowledge Network
net_file = "simulated_data/simulated_prior_data.tsv"
effect_map = {"upregulates-expression": 1, "downregulates-expression": -1}
net = pd.read_csv(net_file, sep="\t", converters={"interaction": effect_map.get}, )
net

In [ ]:
ground_truth_df = pd.read_csv("simulated_data/simulated_ground_truth.tsv", sep="\t", index_col=0)

In [ ]:
def run_z_aggregate(adata, priors: pd.DataFrame, min_targets: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    X = adata.X

    if issparse(X):
        mean, var = mean_variance_axis(X, axis=0)
        std = np.sqrt(var)
    else:
        mean = np.mean(X, axis=0)
        std = np.std(X, axis=0)

    std[std == 0] = 1.0

    # Build Sparse Weight Matrix
    tf_counts = priors["source"].value_counts()
    valid_tfs = tf_counts[tf_counts >= min_targets].index

    if len(valid_tfs) == 0:
        print("Warning: No TFs passed min_targets filter.")
        empty = pd.DataFrame(index=adata.obs_names)
        return empty, empty

    priors_filtered = priors[priors["source"].isin(valid_tfs)]

    # Categorical mapping (already efficient)
    genes_cat = pd.Categorical(priors_filtered["target"], categories=adata.var_names)
    tfs_cat = pd.Categorical(priors_filtered["source"], categories=valid_tfs)

    valid_mask = genes_cat.codes != -1
    row_ind = genes_cat.codes[valid_mask]
    col_ind = tfs_cat.codes[valid_mask]

    # Combine the interaction direction (+1/-1) with the confidence weight (0.0-1.0)
    raw_weights = priors_filtered["weight"].values[valid_mask]
    directions = priors_filtered["interaction"].values[valid_mask]
    # Effective weight = direction * magnitude
    # e.g., Downregulation (-1) with high confidence (0.9) -> -0.9
    data_val = raw_weights * directions

    # W shape: (Genes x TFs)
    W = csr_matrix((data_val, (row_ind, col_ind)), shape=(len(adata.var_names), len(valid_tfs)), dtype=np.float32)

    # Z-Score Matrix Multiplication
    # A. Scale weights by gene standard deviation
    inv_std = (1.0 / std).astype(np.float32)

    # Scale W in-place without creating a diagonal matrix
    W_scaled = W.copy()
    inplace_csr_row_scale(W_scaled, inv_std)

    # B. Compute Terms
    # Term 1: Raw weighted sum
    term1 = X @ W_scaled

    if issparse(term1):
        term1 = term1.toarray()

    # Term 2: Correction for Mean centering
    term2 = mean @ W_scaled

    numerator = term1 - term2

    # Normalization & Stats
    sum_sq_weights = np.array(W.power(2).sum(axis=0)).flatten()

    denominator = np.sqrt(sum_sq_weights)
    denominator[denominator == 0] = 1.0

    final_z = numerator / denominator

    # Fast Statistics
    abs_z = np.abs(final_z)

    # 2 * (1 - cdf(|z|))
    p_values = 2 * ndtr(-abs_z)
    p_values = np.clip(p_values, 1e-300, 1.0)

    activation_dir = np.sign(final_z)
    scores = -np.log(p_values) * activation_dir

    # Return
    scores_df = pd.DataFrame(scores, index=adata.obs_names, columns=valid_tfs)
    pvalues_df = pd.DataFrame(p_values, index=adata.obs_names, columns=valid_tfs)

    return scores_df, pvalues_df



z_aggregate_score_df, _ = run_z_aggregate(
    adata=adata,
    priors=net,
    min_targets=0,
)

# Assign the DataFrame DIRECTLY to obsm
adata.obsm["score_z-aggregate"] = z_aggregate_score_df.loc[adata.obs_names].reindex(sorted(z_aggregate_score_df.columns), axis=1)

# Verify
# print(adata.obsm["score_z-aggregate"].head())

In [ ]:
# Make network decoupler compatible
net_dc = net.copy()
net_dc["weight"] = net_dc["interaction"] * net_dc["weight"]
net_dc = net_dc[["source", "target", "weight"]]

In [ ]:
dc.mt.ulm(adata, net_dc, tmin=0)
dc.mt.mlm(adata, net_dc, tmin=0)
dc.mt.zscore(adata, net_dc, tmin=0)

In [ ]:
sc.pp.scale(adata)
dc.mt.viper(adata, net_dc, tmin=0)

## Plot ROC Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import roc_curve, auc
import seaborn as sns
import os

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'

def plot_custom_benchmarks_paper(adata, ground_truth_df, methods_dict,
                                 text_size=16, filename_prefix=None):
    color_map = {
        "z-aggregate": "#74c476",  # Soft Green
        "ulm": "#6baed6",          # Soft Blue
        "viper": "#ef8a62",        # Salmon/Orange
        "mlm": "#9e9ac8",          # Purple
        "zscore": "#969696"        # Grey
    }

    fallback_palette = sns.color_palette("colorblind", n_colors=10)

    # Define the two tasks
    tasks = [
        {
            "title": "Activated TFs",
            "gt_val": 1,
            "invert_score": False,
            "suffix": "activated"
        },
        {
            "title": "Inhibited TFs",
            "gt_val": -1,
            "invert_score": True,
            "suffix": "inhibited"
        }
    ]

    for task in tasks:
        fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

        # --- Collect Data ---
        roc_data = []
        has_data = False

        for method_name, obsm_key in methods_dict.items():
            if obsm_key not in adata.obsm:
                continue

            scores_df = adata.obsm[obsm_key]
            common_tfs = list(set(ground_truth_df.columns) & set(scores_df.columns))
            common_cells = list(set(ground_truth_df.index) & set(scores_df.index))

            if not common_tfs or not common_cells:
                continue

            # Data prep
            y_true_flat = ground_truth_df.loc[common_cells, common_tfs].values.flatten()
            y_score_flat = scores_df.loc[common_cells, common_tfs].values.flatten()

            # Determine positive class based on task
            if task["gt_val"] == 1:
                y_true_binary = (y_true_flat == 1).astype(int)
            else:
                y_true_binary = (y_true_flat == -1).astype(int)

            if y_true_binary.sum() == 0:
                continue

            has_data = True

            if task["invert_score"]:
                y_score_flat = -y_score_flat

            fpr, tpr, _ = roc_curve(y_true_binary, y_score_flat)
            roc_val = auc(fpr, tpr)

            roc_data.append({
                "name": method_name,
                "fpr": fpr,
                "tpr": tpr,
                "auc": roc_val
            })

        if not has_data:
            plt.close(fig)
            print(f"Skipping {task['title']} - No valid ground truth data found.")
            continue
        # 1. Diagonal random chance line
        ax.plot([0, 1], [0, 1], color='#999999', linestyle='--', lw=1.5, alpha=0.8, zorder=1)

        # 2. Plot Methods
        for i, data in enumerate(roc_data):
            # Determine Color
            c = None
            for key, val in color_map.items():
                if key in data["name"].lower():
                    c = val
                    break
            if c is None:
                c = fallback_palette[i % len(fallback_palette)]

            ax.plot(
                data["fpr"], data["tpr"],
                label=f'{data["name"]} (AUC={data["auc"]:.3f})',
                color=c,
                lw=3.0,
                alpha=0.9,
                zorder=2
            )

        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])

        # Bold/Large Title
        ax.set_title(task["title"], fontsize=text_size + 4, fontweight='bold', pad=15, color='#222222')

        # Labels
        ax.set_xlabel('False Positive Rate', fontsize=text_size, labelpad=10)
        ax.set_ylabel('True Positive Rate', fontsize=text_size, labelpad=10)
        ax.tick_params(axis='both', which='major', labelsize=text_size - 2, width=1, length=5)

        ax.legend(
            loc="lower right",
            fontsize=text_size - 2,
            frameon=False,
            handlelength=1.5,
            borderaxespad=0.5
        )

        ax.grid(True, which='major', color='#dddddd', linestyle='-', linewidth=1.0, alpha=0.8, zorder=0)

        plt.tight_layout()
        if filename_prefix:
            base_prefix = os.path.splitext(filename_prefix)[0]
            fname = f"{base_prefix}_{task['suffix']}.svg"
            plt.savefig(fname, dpi=300, bbox_inches='tight', format='svg')
            print(f"Saved to {fname}")

        plt.show()

# --- Usage Example ---
methods_to_test = {
    "z-aggregate": "score_z-aggregate",
    "viper": "score_viper",
    "ulm": "score_ulm",
    "mlm": "score_mlm",
    "zscore": "score_zscore",
}

plot_custom_benchmarks_paper(
    adata,
    ground_truth_df,
    methods_dict=methods_to_test,
    text_size=16,
    # filename_prefix="illustrations/sim_roc_hard"
)